# Triaje automático de literatura biomédica

### Clasificación de abstracts médicos en cinco áreas clínicas

**Sesión 1 — Embeddings y redes recurrentes** · MIIA, Universidad Icesi

---

Este notebook desarrolla un caso propio sobre las técnicas de los notebooks guía
`3-embeddings-y-lstm-minimo` y `4-lstm-avanzado-con-pytorch-lightning`, pero cambia el
dominio, el corpus y —sobre todo— la pregunta.

La guía clasifica noticias en español y usa los vectores de spaCy únicamente como
demostración de similitud semántica: cuando llega el momento de entrenar, la capa
`nn.Embedding` se inicializa desde cero y los vectores pre-entrenados nunca tocan la red.
Este trabajo cierra ese hueco y lo convierte en el experimento central.

**Contenido de esta entrega (fases 0–2 del plan de trabajo):**

| Sección | Contenido |
|---|---|
| 1 | Planteamiento del problema e hipótesis |
| 2 | Andamiaje reproducible: semillas, split estratificado, métrica única |
| 3 | Análisis exploratorio — siete gráficas, cada una fija una decisión posterior |
| 4 | Síntesis: los hiperparámetros que el EDA deja decididos |

El modelado (baselines, LSTM y ablaciones) continúa en la siguiente entrega.

---

## 1. El problema

### 1.1 Qué se decide

Un *abstract* biomédico llega sin etiquetar y hay que enrutarlo a una de cinco áreas clínicas:

| Etiqueta | Área |
|---|---|
| 1 | Neoplasias |
| 2 | Enfermedades del sistema digestivo |
| 3 | Enfermedades del sistema nervioso |
| 4 | Enfermedades cardiovasculares |
| 5 | Condiciones patológicas generales |

Es el problema real de **triaje de literatura**: PubMed indexa más de un millón de
artículos nuevos al año, y antes de que un revisor humano vea un artículo alguien tiene
que decidir a qué área pertenece. Automatizar ese primer filtro es lo que permite que el
tiempo del revisor se gaste en los casos dudosos y no en los obvios.

### 1.2 Por qué no es "clasificar noticias otra vez"

Tres propiedades del dominio cambian el problema respecto al de los notebooks guía:

**El vocabulario *es* la señal.** La categoría se decide por términos técnicos densos
(`restenosis`, `cholangiocarcinoma`, `electroencephalographic`), no por el tema general
del texto. Un modelo que trunque o descarte ese vocabulario pierde exactamente aquello
que necesita para decidir. Esto tiene una consecuencia directa sobre el uso de embeddings
pre-entrenados que la sección 3.4 convierte en una medición.

**Hay una clase cajón de sastre.** "Condiciones patológicas generales" no es un área
anatómica sino un residuo: agrupa lo que no encaja limpiamente en las otras cuatro. Por
construcción comparte léxico con todas, así que la matriz de confusión no va a ser un
adorno sino el resultado principal.

**Las clases están desbalanceadas.** Como se verifica en §3.1, la clase mayoritaria es un
tercio del corpus. Un clasificador que siempre responda lo mismo acierta un tercio de las
veces sin haber aprendido nada. Reportar únicamente *accuracy* —lo que hacen los dos
notebooks guía— haría que ese modelo vacío pareciera razonable.

### 1.3 Por qué una arquitectura secuencial, y qué la refutaría

El argumento a favor de una LSTM es la **negación y el alcance sintáctico**. En este
dominio abundan construcciones como *"sin evidencia de metástasis"* frente a *"con
evidencia de metástasis"*: una bolsa de palabras las ve idénticas porque contiene los
mismos términos. Un modelo que procesa la secuencia en orden puede, en principio,
distinguirlas.

En principio. Un abstract mediano ronda las 180 palabras, que es una secuencia larga para
una LSTM de una capa, y el desbalance castiga a las clases pequeñas. **Este notebook no
asume que la LSTM gana: la compara contra TF-IDF y deja que los números decidan.** Si un
modelo de bolsa de palabras iguala o supera a la red recurrente, ese es el hallazgo, y es
un hallazgo honesto sobre el costo-beneficio de la arquitectura.

### 1.4 Hipótesis

Se enuncian **antes** de mirar cualquier resultado, para que el análisis posterior sea una
contrastación y no una racionalización:

> **H1 — Valor de la arquitectura secuencial.** Una LSTM supera a TF-IDF + regresión
> logística en macro-F1. *Refutable:* si TF-IDF iguala o gana, el costo computacional de
> la recurrencia no se justifica en este corpus.
>
> **H2 — Valor de los embeddings pre-entrenados.** Inicializar la capa de embedding con
> los vectores de spaCy mejora sobre inicializarla aleatoriamente. *Refutable:* si no hay
> diferencia, el corpus tiene suficiente señal para aprender sus propios vectores.
>
> **H3 — El dominio importa.** El beneficio de H2 se concentra en las clases con menor
> proporción de términos fuera del vocabulario de spaCy. *Refutable:* si la mejora es
> uniforme entre clases, la cobertura léxica no es el mecanismo.

Las tres se contrastan en la siguiente entrega. Este notebook produce la evidencia
descriptiva que las hace medibles.

---

## 2. Andamiaje reproducible

Media hora de infraestructura que se paga sola. Todo lo que viene después compara
resultados entre sí, y sin estas cuatro piezas las comparaciones no significan nada:

1. **Semilla global fija**, para que dos corridas del mismo modelo den el mismo número.
2. **Split estratificado**, porque con clases desbalanceadas 3:1 un `random_split`
   —lo que usan ambos notebooks guía— deja proporciones distintas en cada partición y
   contamina toda comparación posterior.
3. **Una única función de evaluación**, para que ningún experimento se mida distinto.
4. **Un registro persistente**, para que la tabla comparativa final se construya sola.

In [ ]:
import os, re, random, warnings, textwrap
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

DATA = Path("../data/medical_abstracts")
FIGS = Path("figuras");    FIGS.mkdir(exist_ok=True)
RES  = Path("resultados"); RES.mkdir(exist_ok=True)

print(f"semilla={SEED}   dispositivo={DEVICE}")

### 2.1 Estilo y paleta de las figuras

Las siete gráficas comparten una sola definición de estilo. La paleta no se eligió por
gusto: se validó con el criterio de separación perceptual para daltonismo (ΔE ≥ 8 en OKLab
para el par adyacente, contraste ≥ 3:1 contra el fondo).

La regla que se sigue en todo el notebook es que **el color codifica una sola cosa a la
vez**. Cuando una gráfica compara magnitudes entre categorías usa un único tono —el color
no aporta identidad, la etiqueta ya lo hace— y reserva el naranja para señalar el elemento
que la narración está discutiendo.

In [ ]:
AZUL, NARANJA = "#2a78d6", "#eb6834"
TINTA, TINTA_2, MALLA = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "font.family": "DejaVu Sans",
    "axes.edgecolor": MALLA, "axes.linewidth": 1.0,
    "axes.labelcolor": TINTA_2, "axes.titlecolor": TINTA,
    "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlepad": 10, "axes.labelsize": 9.5,
    "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.major.size": 0, "ytick.major.size": 0,
    "grid.color": MALLA, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

def limpiar(ax, ejes=("top", "right")):
    """Quita marcos innecesarios: menos tinta que no es dato."""
    for e in ejes:
        ax.spines[e].set_visible(False)
    return ax

def guardar(fig, nombre):
    fig.savefig(FIGS / f"{nombre}.png")
    return fig

### 2.2 Carga del corpus

El corpus es [`TimSchopf/medical_abstracts`](https://huggingface.co/datasets/TimSchopf/medical_abstracts)
(licencia CC-BY-SA 3.0), derivado de un subconjunto de PubMed. Trae su propia partición
oficial de entrenamiento y prueba, que se respeta: usar la partición del autor en lugar de
inventar una propia mantiene los resultados comparables con cualquier otro trabajo sobre
el mismo dataset.

In [ ]:
# El corpus no se versiona en git (pesa ~10 MB). Si no está en local, se baja del Hub.
BASE_HUB = ("https://huggingface.co/datasets/TimSchopf/medical_abstracts/"
            "resolve/refs%2Fconvert%2Fparquet/")
FUENTES = {"train":  BASE_HUB + "default/train/0000.parquet",
           "test":   BASE_HUB + "default/test/0000.parquet",
           "labels": BASE_HUB + "labels/train/0000.parquet"}

DATA.mkdir(parents=True, exist_ok=True)
for nombre, url in FUENTES.items():
    destino = DATA / f"{nombre}.parquet"
    if not destino.exists():
        print(f"descargando {nombre}…")
        pd.read_parquet(url).to_parquet(destino, index=False)
print(f"corpus disponible en {DATA.resolve()}")

In [ ]:
etiquetas = pd.read_parquet(DATA / "labels.parquet")
NOMBRE = dict(zip(etiquetas.condition_label, etiquetas.condition_name))
CLASES = [NOMBRE[i] for i in sorted(NOMBRE)]

def cargar(particion):
    df = pd.read_parquet(DATA / f"{particion}.parquet")
    df = df.rename(columns={"medical_abstract": "texto", "condition_label": "y"})
    df["clase"] = df.y.map(NOMBRE)
    return df.reset_index(drop=True)

df_oficial_train, df_test = cargar("train"), cargar("test")

print(f"partición oficial de entrenamiento : {len(df_oficial_train):>6,}")
print(f"partición oficial de prueba        : {len(df_test):>6,}")
print(f"clases                             : {len(CLASES)}")
for i, c in sorted(NOMBRE.items()):
    print(f"   {i} · {c}")

In [ ]:
# Un ejemplo, para ver con qué se está trabajando
ej = df_oficial_train.iloc[7]
print(f"CLASE: {ej.clase}\n")
print(textwrap.fill(ej.texto[:700], 92) + " […]")

### 2.3 Split estratificado

Se divide la partición oficial de entrenamiento en **entrenamiento (90%)** y
**validación (10%)**, estratificando por clase. La partición oficial de prueba se guarda
intacta y no se vuelve a mirar hasta el final del modelado.

Dos decisiones que los notebooks guía no toman y que aquí sí importan:

- **`stratify=y`.** Con un desbalance de 3:1, un split aleatorio deja la clase minoritaria
  sobre- o sub-representada en validación por puro azar, y entonces la métrica de
  validación mide el sorteo tanto como el modelo.
- **El EDA se calcula sobre entrenamiento únicamente.** Construir el vocabulario o mirar
  distribuciones sobre validación o prueba es fuga de información: decisiones como el
  tamaño del vocabulario o el `max_length` quedarían ajustadas a datos que se supone que
  el modelo no ha visto.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df_oficial_train, test_size=0.10, stratify=df_oficial_train.y, random_state=SEED
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

reparto = pd.DataFrame({
    "train_%": df_train.clase.value_counts(normalize=True),
    "val_%":   df_val.clase.value_counts(normalize=True),
    "test_%":  df_test.clase.value_counts(normalize=True),
}).mul(100).round(2).loc[CLASES]
reparto["n_train"] = df_train.clase.value_counts().loc[CLASES]

print(f"train={len(df_train):,}   val={len(df_val):,}   test={len(df_test):,}\n")
print("Proporción de cada clase por partición:")
print(reparto.to_string())
print(f"\nDesviación máxima train↔val: "
      f"{(reparto['train_%'] - reparto['val_%']).abs().max():.2f} puntos porcentuales")

La desviación máxima entre entrenamiento y validación queda por debajo de medio punto
porcentual: la estratificación funcionó y las dos particiones son comparables.

### 2.4 La métrica, y por qué no es *accuracy*

Con la clase mayoritaria en un tercio del corpus, el *accuracy* premia a un modelo que
ignore por completo las clases pequeñas. La métrica principal de todo el trabajo es
**macro-F1**, que promedia el F1 de cada clase sin ponderar por su tamaño: equivocarse
sistemáticamente en la clase minoritaria cuesta lo mismo que equivocarse en la mayoritaria.

Se reporta también el *accuracy*, pero como dato secundario y siempre junto al piso de la
clase mayoritaria, para que se lea en su contexto.

La función `evaluar()` es la **única** vía por la que pasa cualquier modelo del proyecto.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, balanced_accuracy_score)

REGISTRO = RES / "experimentos.csv"

def evaluar(y_true, y_pred, nombre, particion="val", notas="", registrar=True, verbose=True):
    """Única función de evaluación del proyecto. Todos los modelos pasan por aquí."""
    m = {
        "experimento":  nombre,
        "particion":    particion,
        "macro_f1":     f1_score(y_true, y_pred, average="macro"),
        "weighted_f1":  f1_score(y_true, y_pred, average="weighted"),
        "accuracy":     accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "notas":        notas,
    }
    if verbose:
        print(f"── {nombre}  ({particion})")
        print(f"   macro-F1 {m['macro_f1']:.4f}   accuracy {m['accuracy']:.4f}   "
              f"bal-acc {m['balanced_acc']:.4f}\n")
        print(classification_report(y_true, y_pred, target_names=CLASES,
                                    digits=3, zero_division=0))
    if registrar:
        pd.DataFrame([m]).to_csv(REGISTRO, mode="a",
                                 header=not REGISTRO.exists(), index=False)
    m["confusion"] = confusion_matrix(y_true, y_pred)
    return m

# Piso de referencia: predecir siempre la clase mayoritaria.
mayoritaria = df_train.y.value_counts().idxmax()
_ = evaluar(df_val.y, np.full(len(df_val), mayoritaria),
            nombre="B0-clase-mayoritaria", particion="val",
            notas=f"predice siempre '{NOMBRE[mayoritaria]}'")

Ahí está el piso, y ahí está el argumento contra el *accuracy* en una sola tabla: el modelo
vacío alcanza un *accuracy* de **0,33** —que suena a algo— con un **macro-F1 de 0,10**, que
es lo que realmente vale. Cualquier modelo posterior tiene que superar holgadamente esa
segunda cifra.

Nótese además el `f1-score` de 0,000 en cuatro de las cinco clases: exactamente el
comportamiento que el *accuracy* agregado esconde.

---

## 3. Análisis exploratorio

El criterio de esta sección es que **cada gráfica justifique una decisión posterior**. No
hay gráficas de adorno: la distribución de clases fija la métrica, el histograma de
longitudes fija `max_length`, la curva de cobertura fija el tamaño del vocabulario, y la
medición de OOV decide si tiene sentido siquiera intentar el experimento de embeddings
pre-entrenados.

Todo se calcula **sobre `df_train`**, nunca sobre validación o prueba.

### 3.1 Distribución de clases → fija la métrica

In [ ]:
conteo = df_train.clase.value_counts().loc[CLASES].sort_values()
piso = conteo.max() / conteo.sum()

fig, ax = plt.subplots(figsize=(8.4, 3.4))
colores = [NARANJA if c == conteo.idxmax() else AZUL for c in conteo.index]
ax.barh(range(len(conteo)), conteo.values, color=colores, height=0.62)

ax.set_yticks(range(len(conteo)))
ax.set_yticklabels([c.replace(" diseases", "").replace(" conditions", "") for c in conteo.index])
for i, v in enumerate(conteo.values):
    ax.text(v + 40, i, f"{v:,}   {v/conteo.sum():.1%}", va="center", fontsize=9, color=TINTA_2)

ax.set_xlim(0, conteo.max() * 1.32)
ax.set_title("Las clases están desbalanceadas 3,2 a 1")
ax.xaxis.set_visible(False)
limpiar(ax, ("top", "right", "bottom"))
ax.text(0, -0.95, f"piso de la clase mayoritaria = {piso:.1%} de accuracy",
        fontsize=8.5, color=NARANJA, style="italic")
guardar(fig, "01-distribucion-clases"); plt.show()

print(f"ratio mayoritaria/minoritaria = {conteo.max()/conteo.min():.2f}")
print(f"piso de accuracy              = {piso:.4f}")

**Lectura.** El desbalance es moderado pero suficiente para invalidar el *accuracy*: la
clase mayoritaria, *general pathological conditions*, concentra un tercio de los datos y
triplica a *digestive system diseases*.

Esto no es un accidente del muestreo, es una propiedad del esquema de etiquetado. Las otras
cuatro clases son áreas anatómicas bien delimitadas; la mayoritaria es el residuo que recoge
lo que no cae limpiamente en ninguna. Un residuo grande y heterogéneo es justamente el tipo
de clase que un modelo aprende a usar como respuesta por defecto, lo que infla el *accuracy*
mientras hunde el F1 de las clases pequeñas.

> **Decisión que fija:** macro-F1 como métrica principal, y pérdida ponderada por clase
> como variable a probar en la fase de modelado.

### 3.2 Longitud de los textos → fija `max_length`

El notebook 4 de la guía mide la longitud **en caracteres** y luego elige un `max_length`
**en tokens**, mezclando dos unidades: el markdown dice "2000 tokens", el siguiente dice
2048 y el código usa 512, sin que ningún número se derive de otro.

Aquí se mide en la unidad correcta —tokens, con el mismo tokenizador que va a usar el
modelo— y el corte se elige mirando cuánto texto se pierde con cada opción.

In [ ]:
def tokenizar(texto):
    """Mismo tokenizador que usará el modelo: minúsculas, alfanumérico."""
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).split()

df_train["tokens"] = df_train.texto.map(tokenizar)
df_train["n_tok"]  = df_train.tokens.map(len)
n_tok = df_train.n_tok

corte = pd.DataFrame([
    {"max_length": L,
     "docs_truncados_%":  (n_tok > L).mean() * 100,
     "tokens_perdidos_%": np.maximum(n_tok - L, 0).sum() / n_tok.sum() * 100}
    for L in (64, 128, 192, 256, 320, 384, 512)
]).round(1)

print(n_tok.describe(percentiles=[.05, .25, .5, .75, .9, .95, .99]).round(1).to_string())
print()
print(corte.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.6),
                               gridspec_kw={"width_ratios": [1.5, 1]})

# — izquierda: distribución con los cortes candidatos
ax1.hist(n_tok, bins=70, color=AZUL, alpha=.85, edgecolor="white", linewidth=.4)
tope = ax1.get_ylim()[1]
ax1.set_ylim(0, tope * 1.24)          # espacio libre arriba para las etiquetas

# Los cortes candidatos. Las líneas se recortan justo debajo de la banda de etiquetas
# (ymax en fracción de eje) para que ninguna atraviese su propio rótulo.
TECHO = 1 / 1.24
for L, est in [(128, ":"), (256, "-"), (384, "--")]:
    ax1.axvline(L, ymax=TECHO, color=NARANJA, linestyle=est, linewidth=1.6)
    ax1.text(L, tope * 1.04, str(L), color=NARANJA, fontsize=9,
             fontweight="bold", ha="center")
# la mediana, una fila más arriba: así nunca choca con la etiqueta de 128
ax1.axvline(n_tok.median(), ymax=TECHO, color=TINTA_2, linewidth=1.2)
ax1.text(n_tok.median(), tope * 1.15, f"mediana {int(n_tok.median())}",
         color=TINTA_2, fontsize=8.5, ha="center")

ax1.set_xlabel("tokens por abstract"); ax1.set_ylabel("abstracts")
ax1.set_title("Distribución de longitudes y cortes candidatos")
ax1.set_xlim(0, 640); ax1.grid(axis="y"); ax1.set_axisbelow(True); limpiar(ax1)

# — derecha: el costo real de cada corte
x = np.arange(len(corte)); ancho = .38
ax2.bar(x - ancho/2, corte["docs_truncados_%"],  ancho, color=AZUL,    label="documentos truncados")
ax2.bar(x + ancho/2, corte["tokens_perdidos_%"], ancho, color=NARANJA, label="tokens perdidos")
ax2.set_xticks(x); ax2.set_xticklabels(corte.max_length, fontsize=8.5)
ax2.set_xlabel("max_length"); ax2.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax2.set_title("Qué cuesta cada corte")
ax2.legend(loc="upper right"); ax2.grid(axis="y"); ax2.set_axisbelow(True); limpiar(ax2)

plt.tight_layout(); guardar(fig, "02-longitudes"); plt.show()

**Lectura.** La distribución es unimodal y con cola derecha corta: mediana de 181 tokens,
percentil 99 en 395, máximo 622. No hay documentos patológicamente largos que obliguen a
truncar agresivamente.

El panel derecho es el que decide, y muestra por qué las dos métricas de truncamiento
cuentan historias distintas. Con `max_length=256`, **una quinta parte de los documentos se
trunca, pero sólo se pierde un 4,8% de los tokens**: lo que se corta son las colas de los
abstracts más largos, no textos completos. Bajar a 128 multiplicaría por más de siete ese
costo, hasta el 36,6% de los tokens; subir a 384 lo lleva casi a cero pero multiplica por
1,5 el cómputo de una LSTM, cuyo costo crece linealmente con la longitud de la secuencia.

> **Decisión que fija:** `max_length = 256` como valor base, con 128 y 384 como ablación A4
> para verificar empíricamente si ese 4,8% de tokens perdidos afecta la métrica.

### 3.3 Vocabulario → fija `max_vocab`

In [ ]:
frec = Counter()
for t in df_train.tokens:
    frec.update(t)

total_tokens, n_tipos = sum(frec.values()), len(frec)
hapax = sum(1 for v in frec.values() if v == 1)
ordenadas = np.array(sorted(frec.values(), reverse=True))
cobertura_acum = np.cumsum(ordenadas) / total_tokens

print(f"tokens totales  : {total_tokens:>9,}")
print(f"tipos distintos : {n_tipos:>9,}")
print(f"hapax legomena  : {hapax:>9,}   ({hapax/n_tipos:.1%} de los tipos, "
      f"{hapax/total_tokens:.2%} de los tokens)")
print()
for V in (2_000, 5_000, 10_000, 20_000, 30_000):
    print(f"  vocabulario de {V:>6,} tipos  →  cubre {cobertura_acum[min(V, n_tipos)-1]:.2%} de los tokens")
print(f"\n12 tipos más frecuentes: {[w for w, _ in frec.most_common(12)]}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.6))

# — izquierda: cobertura acumulada del corpus
ax1.plot(np.arange(1, n_tipos + 1), cobertura_acum * 100, color=AZUL, linewidth=2)
# las dos anotaciones se escalonan a lados opuestos para que no se pisen
for V, desplaza in [(5_000, (-104, -40)), (20_000, (14, -14))]:
    c = cobertura_acum[V - 1] * 100
    ax1.plot([V], [c], "o", color=NARANJA, markersize=8,
             markeredgecolor="white", markeredgewidth=1.5)
    ax1.annotate(f"{V:,} tipos\n{c:.1f}% de tokens", (V, c), textcoords="offset points",
                 xytext=desplaza, fontsize=8.5, color=TINTA_2)
ax1.set_xscale("log"); ax1.set_xlabel("tamaño del vocabulario (escala log)")
ax1.set_ylabel("cobertura del corpus"); ax1.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax1.set_ylim(0, 103); ax1.set_title("Un vocabulario pequeño cubre casi todo")
ax1.grid(); ax1.set_axisbelow(True); limpiar(ax1)

# — derecha: Zipf
rango = np.arange(1, n_tipos + 1)
ax2.loglog(rango, ordenadas, color=AZUL, linewidth=1.8, label="corpus")
ax2.loglog(rango, ordenadas[0] / rango, color=NARANJA, linestyle="--", linewidth=1.4,
           label="Zipf ideal  (f ∝ 1/rango)")
ax2.set_xlabel("rango del tipo"); ax2.set_ylabel("frecuencia")
ax2.set_title("El corpus se comporta como lenguaje natural")
ax2.legend(loc="upper right"); ax2.grid(which="major"); ax2.set_axisbelow(True); limpiar(ax2)

plt.tight_layout(); guardar(fig, "03-vocabulario"); plt.show()

**Lectura.** El corpus sigue una ley de Zipf casi de manual —la curva empírica se pega a la
referencia `f ∝ 1/rango` durante tres órdenes de magnitud— con la desviación típica en la
cola: los tipos raros caen más rápido de lo que predice Zipf, que es lo que se espera de un
corpus de dominio cerrado y vocabulario técnico.

La curva de cobertura de la izquierda es la que decide. **Un vocabulario de 20.000 tipos
cubre casi el 99% de los tokens**, y duplicarlo apenas añadiría un punto: los *hapax
legomena* son una cuarta parte de los tipos distintos pero menos del medio por ciento de
los tokens. Cada uno de ellos, si entrara al vocabulario, sería una fila de la matriz de
embeddings entrenada con un único ejemplo — ruido con coste de memoria.

> **Decisión que fija:** `max_vocab = 20.000`, con `[UNK]` absorbiendo el resto.

### 3.4 Cobertura de vectores pre-entrenados → el hallazgo central

Esta es la sección que conecta el notebook con su hipótesis principal. El notebook 3 de la
guía termina su recorrido por los vectores de spaCy con una advertencia:

> *"si sabemos de antemano que nuestro corpus tiene tokens que no están en el vocabulario,
> pues lo más seguro es que nuestro modelo no sea lo suficientemente bueno"*

...y lo ilustra con una palabra inventada, `nargle`. Nunca mide la tasa real sobre un
corpus real. En un dominio técnico esa medición no es un detalle: es la que determina si
tiene sentido siquiera intentar inicializar la red con vectores pre-entrenados.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_lg")
print(f"vocabulario de en_core_web_lg: {nlp.vocab.vectors.shape[0]:,} vectores "
      f"de {nlp.vocab.vectors.shape[1]} dimensiones")

tiene_vector = {w: nlp.vocab[w].has_vector for w in frec}
cob_tipo  = float(np.mean(list(tiene_vector.values())))
cob_token = sum(c for w, c in frec.items() if tiene_vector[w]) / total_tokens
sin_vector = [w for w, _ in frec.most_common(4000) if not tiene_vector[w]]

print(f"\nCobertura sobre el corpus de entrenamiento")
print(f"  por TIPO de palabra : {cob_tipo:>6.1%}   "
      f"({sum(1 for v in tiene_vector.values() if not v):,} tipos sin vector)")
print(f"  por TOKEN           : {cob_token:>6.1%}")
print(f"\nTérminos frecuentes SIN vector (los 20 más comunes del corpus):")
print(textwrap.fill(", ".join(sin_vector[:20]), 92))

**El resultado tiene dos caras, y ahí está lo interesante.** Sólo dos de cada tres tipos de
palabra distintos tienen vector en spaCy, pero el 97% de los tokens que aparecen sí lo
tienen. No hay contradicción: los términos sin vector son los raros.

El problema es *cuáles* son. La lista no es ruido ni erratas — son `restenosis`,
`echocardiographic`, `transesophageal`, `immunohistochemical`, `normotensive`, `stenoses`.
Es decir, **exactamente la terminología que discrimina entre áreas clínicas**. Los términos
que spaCy sí cubre son los que comparten todos los abstracts (`patients`, `study`,
`treatment`, `results`), que son los que menos ayudan a separar clases.

Esa asimetría es la que hace que H2 no sea obvia en ninguna dirección, y es lo que se mide
a continuación por clase.

#### Cobertura por clase: cuidado con un artefacto de conteo

La comparación ingenua —contar los tipos de cada clase y medir qué fracción tiene vector—
**está confundida por el tamaño de la clase**. Una clase con más documentos acumula más
términos raros, y los términos raros son precisamente los que no tienen vector: la métrica
ingenua mide en buena parte cuántos documentos tiene cada clase.

La corrección es submuestrear todas las clases al tamaño de la más pequeña y repetir el
remuestreo varias veces. Se calculan las dos versiones para dejar el artefacto a la vista.

In [ ]:
n_min = df_train.clase.value_counts().min()
filas = []
for clase, g in df_train.groupby("clase"):
    c_todo = Counter()
    for t in g.tokens:
        c_todo.update(t)
    ingenua = np.mean([tiene_vector[w] for w in c_todo])

    muestras = []
    for s in range(20):
        c_sub = Counter()
        for t in g.sample(n_min, random_state=s).tokens:
            c_sub.update(t)
        muestras.append(np.mean([tiene_vector[w] for w in c_sub]))

    filas.append({"clase": clase, "n_docs": len(g), "tipos": len(c_todo),
                  "ingenua_%": ingenua * 100,
                  "balanceada_%": float(np.mean(muestras)) * 100,
                  "sd": float(np.std(muestras)) * 100})

cob = pd.DataFrame(filas).sort_values("balanceada_%").reset_index(drop=True)
print(f"Submuestreo a {n_min:,} documentos por clase, 20 remuestreos\n")
print(cob.round(2).to_string(index=False))
print(f"\nrango ingenuo    : {cob['ingenua_%'].max() - cob['ingenua_%'].min():.1f} puntos")
print(f"rango balanceado : {cob['balanceada_%'].max() - cob['balanceada_%'].min():.1f} puntos")
print(f"correlación (n_docs, cobertura ingenua) = {cob.n_docs.corr(cob['ingenua_%']):+.3f}")
print(f"peor clase según la métrica ingenua    : {cob.sort_values('ingenua_%').clase.iloc[0]}")
print(f"peor clase según la métrica balanceada : {cob.clase.iloc[0]}")

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 3.6))
y = np.arange(len(cob)); ancho = .36

ax.barh(y + ancho/2, cob["ingenua_%"], ancho, color=AZUL,
        label="ingenua (todos los documentos de la clase)")
ax.barh(y - ancho/2, cob["balanceada_%"], ancho, color=NARANJA,
        xerr=cob["sd"], error_kw={"ecolor": TINTA_2, "elinewidth": 1, "capsize": 2},
        label=f"balanceada ({n_min:,} docs/clase, 20 remuestreos)")

ax.set_yticks(y)
ax.set_yticklabels([c.replace(" diseases", "").replace(" conditions", "") for c in cob.clase])
ax.set_xlim(60, 92); ax.set_xlabel("tipos de palabra con vector en spaCy")
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_title("El mismo dato, medido de dos maneras, cambia de orden")
# la leyenda va debajo del eje: dentro del área de trazado pisaba las barras inferiores
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.26), ncol=2)
ax.grid(axis="x"); ax.set_axisbelow(True)
limpiar(ax, ("top", "right"))
plt.tight_layout(); guardar(fig, "04-cobertura-oov"); plt.show()

**Lectura — y este es el hallazgo metodológico del notebook.** Las dos mediciones no sólo
difieren en magnitud: **cambian el orden de las clases**.

Con el conteo ingenuo, *general pathological conditions* parece la clase con peor cobertura
léxica (74,6%), lo que encajaría cómodamente con la narrativa de "es la clase cajón de
sastre, tiene el vocabulario más disperso". Al controlar por número de documentos esa clase
sube a 82,2% y **el puesto de peor cobertura pasa a *neoplasms***. La correlación entre
número de documentos y cobertura ingenua es de **−0,886**: la firma inequívoca del
artefacto.

Además, el rango entre clases se comprime **de 8,8 a 4,4 puntos, exactamente a la mitad**.
Es decir: **la mitad de la variación aparente entre clases era tamaño de muestra, no
especificidad del vocabulario.**

Esto tiene una consecuencia directa sobre H3. Si la diferencia real de cobertura entre
clases es de sólo unos pocos puntos, el efecto diferencial que H3 predice va a ser pequeño,
y habrá que medirlo con cuidado —varias semillas, dispersión reportada— en vez de leer una
única corrida. Es exactamente el tipo de expectativa que conviene fijar antes de entrenar,
para no confundir ruido con confirmación.

> **Decisión que fija:** el experimento de embeddings (E2/E3) se corre con varias semillas y
> se reporta la media con su dispersión. H3 se contrasta contra la cobertura *balanceada*,
> no la ingenua.

### 3.5 Términos discriminativos → qué debería aprender el modelo

Ordenar por frecuencia bruta sólo devuelve palabras funcionales (`the`, `of`, `and`) y
términos genéricos del género (`patients`, `study`). Para ver qué distingue *realmente* a
cada clase se usa el **log-odds ratio con prior Dirichlet informado** (Monroe, Colaresi &
Quinn, 2008), que compara la frecuencia de un término dentro de la clase contra el resto
del corpus y la normaliza por su varianza, de modo que un término raro necesita una
diferencia mucho mayor para destacar.

In [ ]:
V = [w for w, c in frec.items() if c >= 20]           # descarta la cola de ruido
alpha = np.array([frec[w] for w in V], dtype=float)   # prior informado por el corpus
a0 = alpha.sum()

conteos = {}
for clase, g in df_train.groupby("clase"):
    c = Counter()
    for t in g.tokens:
        c.update(t)
    conteos[clase] = np.array([c.get(w, 0) for w in V], dtype=float)
total_v = sum(conteos.values())

def log_odds(clase, k=12):
    """z-score del log-odds ratio de cada término: clase vs. resto del corpus."""
    yi = conteos[clase]; yj = total_v - yi
    ni, nj = yi.sum(), yj.sum()
    delta = (np.log((yi + alpha) / (ni + a0 - yi - alpha))
             - np.log((yj + alpha) / (nj + a0 - yj - alpha)))
    z = delta / np.sqrt(1 / (yi + alpha) + 1 / (yj + alpha))
    idx = np.argsort(-z)[:k]
    return [V[i] for i in idx], z[idx]

print(f"vocabulario evaluado: {len(V):,} tipos con frecuencia ≥ 20\n")
for clase in CLASES:
    palabras, z = log_odds(clase, 10)
    print(f"{clase[:33]:33s} │ z_max={z[0]:5.1f} │ {', '.join(palabras)}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14.5, 4.0), sharex=True)
for ax, clase in zip(axes, CLASES):
    palabras, z = log_odds(clase, 10)
    orden = np.argsort(z)
    ax.barh(range(10), z[orden], color=AZUL, height=.68)
    ax.set_yticks(range(10))
    ax.set_yticklabels([palabras[i] for i in orden], fontsize=8.5)
    ax.set_title(clase.replace(" diseases", "").replace(" conditions", ""), fontsize=9.5)
    ax.grid(axis="x"); ax.set_axisbelow(True); limpiar(ax, ("top", "right"))
axes[0].set_xlabel("z del log-odds ratio")
fig.suptitle("Términos que más distinguen a cada clase del resto del corpus",
             x=.005, ha="left", fontsize=11.5, fontweight="semibold", color=TINTA)
plt.tight_layout(rect=[0, 0, 1, .93]); guardar(fig, "05-terminos-discriminativos"); plt.show()

**Lectura.** Las cuatro clases anatómicas producen listas que un médico firmaría sin dudar:
*cardiovascular* se define por `coronary`, `ventricular`, `myocardial`; *digestive* por
`hepatitis`, `cirrhosis`, `biliary`; *nervous* por `brain`, `cerebral`, `stroke`;
*neoplasms* por `cancer`, `carcinoma`, `chemotherapy`. Términos específicos, de alta
frecuencia dentro de su clase y casi ausentes fuera.

**La clase mayoritaria es la excepción, y confirma lo que se sospechaba en §1.2.** Sus
términos más discriminativos —`postoperative`, `infection`, `children`, `graft`,
`bleeding`— no describen un sistema anatómico sino *estados clínicos transversales*: cosas
que le pasan a un paciente independientemente del órgano afectado.

Y hay una segunda lectura, que el eje x compartido entre los cinco paneles hace evidente:
**su z máximo es 7,1, frente a 32,2 de *neoplasms* y 28,2 de *cardiovascular***. No es que
sus términos sean distintos — es que ninguno la identifica con fuerza. Las barras del
último panel son visiblemente más cortas que las del primero.

Eso es la definición operativa de una clase residual, y permite una **predicción concreta**:
los errores del modelo no van a repartirse al azar, van a concentrarse en confundir esta
clase con las otras cuatro. La sección siguiente cuantifica esa predicción.

Un detalle práctico que también sale de aquí: `0` aparece entre los términos discriminativos
de *cardiovascular*, residuo de valores numéricos (`0.05`, `p < 0.001`) que el tokenizador
parte. Es un candidato claro para normalizar a un token `[NUM]` en la fase de modelado.

### 3.6 Similitud entre clases → predice la matriz de confusión

Última pieza del EDA, y la más apostada: **predecir dónde se va a equivocar el modelo antes
de entrenarlo.**

El procedimiento es representar cada clase por el centroide de sus documentos en espacio
TF-IDF y medir la similitud coseno entre centroides — la misma métrica que el notebook 3
usa para comparar `lion`, `cat` y `pet`, aplicada aquí a clases enteras en lugar de palabras
sueltas. Dos clases con centroides muy próximos comparten vocabulario, y un clasificador
que se apoye en el vocabulario debería confundirlas.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(min_df=5, max_df=.5, sublinear_tf=True, stop_words="english")
X = vec.fit_transform(df_train.texto)
centroides = np.vstack([np.asarray(X[(df_train.clase == c).values].mean(axis=0)) for c in CLASES])
S = cosine_similarity(centroides)

corto = [c.replace(" diseases", "").replace(" conditions", "") for c in CLASES]

# La diagonal vale 1 por construcción: dejarla dentro aplastaría toda la escala de color
# contra el rango que sí importa (58–88). Se enmascara y el color se reescala a los datos.
S_off = S.copy()
np.fill_diagonal(S_off, np.nan)
vmin, vmax = np.nanmin(S_off), np.nanmax(S_off)

import matplotlib as mpl
mapa = mpl.colormaps["Blues"].with_extremes(bad="#f0efec")

fig, ax = plt.subplots(figsize=(6.6, 5.3))
im = ax.imshow(S_off, cmap=mapa, vmin=vmin, vmax=vmax)
ax.set_xticks(range(5)); ax.set_xticklabels(corto, rotation=32, ha="right", fontsize=9)
ax.set_yticks(range(5)); ax.set_yticklabels(corto, fontsize=9)
umbral = vmin + (vmax - vmin) * .58          # a partir de aquí el fondo pide texto claro
for i in range(5):
    for j in range(5):
        if i == j:
            ax.text(j, i, "—", ha="center", va="center", fontsize=11, color="#b9b7b1")
        else:
            ax.text(j, i, f"{S[i, j]*100:.0f}", ha="center", va="center", fontsize=9.5,
                    color="white" if S[i, j] > umbral else TINTA)
ax.set_title("Similitud coseno entre centroides TF-IDF")
for e in ("top", "right", "bottom", "left"):
    ax.spines[e].set_visible(False)
cb = fig.colorbar(im, ax=ax, shrink=.72); cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=8.5)
cb.ax.yaxis.set_major_formatter(lambda v, _: f"{v*100:.0f}%")
plt.tight_layout(); guardar(fig, "06-similitud-clases"); plt.show()

iu = np.triu_indices(5, 1)
pares = sorted(zip(S[iu], [(corto[i], corto[j]) for i, j in zip(*iu)]), reverse=True)
print("Confusión esperada, de mayor a menor similitud:")
for s, (a, b) in pares:
    print(f"  {s*100:5.1f}%   {a}  ↔  {b}")

**Lectura y predicción registrada.** El patrón es inequívoco: *general pathological
conditions* es la clase más parecida a **todas** las demás, mientras que los pares entre
clases anatómicas son claramente más distantes. El par más disímil, *neoplasms* ↔
*cardiovascular*, corresponde a las dos áreas con menos solapamiento clínico real.

Esto convierte el heatmap en una hipótesis falsable sobre una matriz de confusión que
todavía no se ha calculado:

> **P1.** La fila y la columna de *general pathological conditions* concentrarán la mayoría
> de los errores del modelo.
>
> **P2.** El par más confundido será *general* ↔ *nervous system*.
>
> **P3.** El par *neoplasms* ↔ *cardiovascular* será el mejor separado, con error casi nulo
> entre ambos.

En la siguiente entrega se superpone la matriz de confusión real sobre esta matriz de
similitud. Si las tres predicciones se cumplen, queda demostrado que el modelo se apoya
esencialmente en evidencia léxica — lo que a su vez sería un argumento **en contra** de H1,
porque es justo lo que un TF-IDF hace bien y sin necesidad de recurrencia.

---

## 4. Síntesis

### 4.1 Lo que el EDA deja decidido

Ninguna de estas decisiones es una convención heredada del notebook guía: cada una sale de
una medición de la sección 3.

| Decisión | Valor | Evidencia |
|---|---|---|
| Métrica principal | macro-F1 | §3.1 — desbalance 3,2:1; el piso de *accuracy* es 33,3% con macro-F1 de 0,10 |
| `max_length` | 256 tokens | §3.2 — trunca el 20% de documentos pero sólo el 4,8% de los tokens |
| `max_vocab` | 20.000 tipos | §3.3 — cubre ~99% de los tokens; una cuarta parte de los tipos son *hapax* |
| Normalización | `[NUM]` para dígitos | §3.5 — `0` aparece como término "discriminativo" espurio |
| Diseño de E2/E3 | varias semillas + dispersión | §3.4 — el rango real de cobertura entre clases es pequeño |
| Validación de H3 | contra cobertura *balanceada* | §3.4 — la métrica ingenua está confundida con el tamaño de clase |

### 4.2 Hallazgos que van más allá de lo pedido

**El artefacto de conteo en la cobertura por clase (§3.4).** La medición ingenua no sólo
exagera las diferencias entre clases: invierte su orden. Es un recordatorio de que una
métrica calculada sobre conjuntos de tamaños distintos casi nunca compara lo que parece
comparar, y de que la corrección —submuestrear y remuestrear— cuesta diez líneas.

**La asimetría entre cobertura por tipo y por token (§3.4).** El 97% por token sugiere que
los vectores de spaCy bastan; el 67% por tipo, junto con la inspección de *qué* términos
faltan, sugiere lo contrario. Ambas cifras son correctas y describen fenómenos distintos.
Reportar sólo una de las dos —que es lo que invita a hacer la advertencia del notebook
guía— daría una conclusión equivocada en cualquiera de las dos direcciones.

**La predicción registrada de la matriz de confusión (§3.6).** Publicar P1–P3 antes de
entrenar convierte la siguiente entrega en una contrastación en lugar de una descripción.

### 4.3 De aquí en adelante

El EDA deja el terreno preparado y tres preguntas abiertas. La sección 5 construye los
modelos que las responden, empezando por el rival honesto (TF-IDF) antes que por la red.

---

## 5. Baselines y LSTM de referencia

Tres modelos, en orden de ambición creciente:

| ID | Modelo | Qué responde |
|---|---|---|
| `B0` | Clase mayoritaria | El piso. Ya calculado en §2.4: macro-F1 0,0998 |
| `B1` | TF-IDF + regresión logística | ¿La recurrencia justifica su costo? **(H1)** |
| `E1` | LSTM con embedding aleatoria | La referencia neuronal, sobre la que se montan E2–E4 |

`E1` replica deliberadamente el diseño del notebook 3 de la guía —una capa de embedding
entrenada desde cero, una LSTM de una capa, y la clasificación a partir del último estado
oculto— porque su papel es ser el punto de partida honesto contra el que se miden las
mejoras posteriores. Los defectos de ese diseño (el estado oculto contaminado por
*padding*, en particular) se corrigen en la fase 5 como ablaciones medidas, no de tapadillo.

Lo que sí se corrige desde ya son los errores que invalidarían la comparación: pérdida
sobre logits crudos sin doble `log_softmax`, *early stopping* sobre la métrica de
validación y no sobre la de entrenamiento, y varias semillas en lugar de una sola corrida.

### 5.1 B1 — TF-IDF y regresión logística

El rival a batir. Entrena en segundos y en clasificación temática es notoriamente difícil
de superar. Se prueban dos variantes para aislar el efecto del desbalance: la versión
estándar y otra con `class_weight="balanced"`, que pondera cada clase por el inverso de su
frecuencia.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import time

def construir_tfidf(balanceado):
    return make_pipeline(
        TfidfVectorizer(min_df=3, max_df=.5, sublinear_tf=True,
                        ngram_range=(1, 2), stop_words="english"),
        LogisticRegression(max_iter=2000, C=4.0, random_state=SEED,
                           class_weight="balanced" if balanceado else None),
    )

resultados_b1 = {}
for balanceado in (False, True):
    etiqueta = "B1-tfidf-balanceado" if balanceado else "B1-tfidf"
    t0 = time.perf_counter()
    modelo = construir_tfidf(balanceado).fit(df_train.texto, df_train.y)
    seg = time.perf_counter() - t0
    pred = modelo.predict(df_val.texto)
    resultados_b1[etiqueta] = {
        "modelo": modelo, "pred": pred,
        "m": evaluar(df_val.y, pred, nombre=etiqueta, particion="val",
                     notas=f"{seg:.1f}s de entrenamiento", verbose=balanceado),
        "seg": seg,
    }
    print(f"[{etiqueta}] macro-F1 {resultados_b1[etiqueta]['m']['macro_f1']:.4f}  "
          f"({seg:.1f} s)\n")

### 5.2 Preparación de datos para la red

Aquí se aplican, sin excepción, las decisiones que fijó el EDA: vocabulario de 20.000 tipos
construido **sólo sobre entrenamiento**, `max_length` de 256, y normalización de dígitos a
`[NUM]` para eliminar el término espurio detectado en §3.5.

Se guarda además la longitud real de cada secuencia. E1 no la usa —replica el diseño de la
guía— pero las ablaciones de la fase 5 la necesitan para enmascarar el *padding*.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

MAX_LEN, MAX_VOCAB = 256, 20_000
PAD, UNK, NUM = "[PAD]", "[UNK]", "[NUM]"

def tokenizar_modelo(texto):
    """Tokenizador de §3.2 más la normalización de dígitos decidida en §3.5."""
    return [NUM if t.isdigit() else t for t in tokenizar(texto)]

# El vocabulario se construye SOLO con entrenamiento: mirarlo en val o test sería fuga.
frec_modelo = Counter()
for t in df_train.texto.map(tokenizar_modelo):
    frec_modelo.update(t)

vocab = {PAD: 0, UNK: 1}
for palabra, _ in frec_modelo.most_common(MAX_VOCAB - 2):
    vocab[palabra] = len(vocab)
id_a_token = {i: w for w, i in vocab.items()}

cubiertos = sum(c for w, c in frec_modelo.items() if w in vocab)
print(f"vocabulario : {len(vocab):,} tipos")
print(f"cobertura   : {cubiertos/sum(frec_modelo.values()):.2%} de los tokens de entrenamiento")
print(f"'{NUM}' ocupa la posición {list(vocab).index(NUM)} por frecuencia")

def codificar(texto, max_len=MAX_LEN):
    ids = [vocab.get(t, 1) for t in tokenizar_modelo(texto)[:max_len]]
    n = len(ids)
    return ids + [0] * (max_len - n), n

class Abstracts(Dataset):
    def __init__(self, df, max_len=MAX_LEN):
        pares = [codificar(t, max_len) for t in df.texto]
        self.ids  = torch.tensor([p[0] for p in pares], dtype=torch.long)
        self.lens = torch.tensor([p[1] for p in pares], dtype=torch.long)
        # las etiquetas del corpus van de 1 a 5; el modelo trabaja con 0..4
        self.y    = torch.tensor(df.y.values - 1, dtype=torch.long)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.ids[i], self.lens[i], self.y[i]

ds_train, ds_val, ds_test = Abstracts(df_train), Abstracts(df_val), Abstracts(df_test)
print(f"\ntensores: train {tuple(ds_train.ids.shape)}  val {tuple(ds_val.ids.shape)}  "
      f"test {tuple(ds_test.ids.shape)}")
print(f"tokens de padding en entrenamiento: "
      f"{(ds_train.ids == 0).float().mean():.1%} de todas las posiciones")

Ese último número merece una pausa: **casi un tercio de las posiciones que la red va a
procesar son relleno**. En el diseño de la guía —tomar `hidden[-1]`— la LSTM llega al final
de la secuencia habiendo procesado, en promedio, decenas de pasos de `[PAD]` después del
último token real. El estado que se usa para clasificar es el que quedó *después* de ese
recorrido en vacío.

Es una debilidad concreta y medible, y la fase 5 la cuantifica. Aquí se deja tal cual,
porque E1 tiene que ser una réplica fiel para que la comparación signifique algo.

Un detalle lateral que confirma la decisión de §3.5: al colapsar todos los dígitos, **`[NUM]`
queda como el token más frecuente del corpus**, por delante de `the`. Los abstracts
biomédicos están saturados de cifras —dosis, tamaños de muestra, valores de *p*— y sin
normalizar habrían ocupado miles de entradas del vocabulario, cada una con su propio vector
entrenado desde un puñado de apariciones.

### 5.3 E1 — LSTM con embedding aleatoria

Arquitectura mínima: embedding entrenable → LSTM de una capa → capa lineal. La pérdida se
calcula sobre **logits crudos**; `nn.CrossEntropyLoss` ya aplica `log_softmax` internamente,
así que añadir una capa `LogSoftmax` antes —como hace el notebook 4 de la guía— la aplicaría
dos veces y distorsionaría el gradiente.

In [ ]:
class LSTMClasificador(nn.Module):
    """Réplica del diseño del notebook 3 de la guía: clasifica desde el último estado oculto."""

    def __init__(self, vocab_size, num_classes=5, emb_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.salida = nn.Linear(hidden_dim, num_classes)

    def forward(self, ids, lens=None):
        _, (h, _) = self.lstm(self.embedding(ids))
        return self.salida(h[-1])          # logits crudos, sin softmax

def semillar(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predecir(modelo, loader):
    modelo.eval()
    logits, reales = [], []
    for ids, lens, y in loader:
        logits.append(modelo(ids.to(DEVICE), lens.to(DEVICE)).cpu())
        reales.append(y)
    return torch.cat(logits), torch.cat(reales)

def entrenar(construir_modelo, semilla, epocas=25, paciencia=4, lr=1e-3,
             batch=64, etiqueta="modelo"):
    """Entrena con early stopping sobre macro-F1 de validación y restaura el mejor estado."""
    semillar(semilla)
    g = torch.Generator().manual_seed(semilla)
    dl_train = DataLoader(ds_train, batch_size=batch, shuffle=True, generator=g, num_workers=0)
    dl_val   = DataLoader(ds_val,   batch_size=256, num_workers=0)

    modelo = construir_modelo().to(DEVICE)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    criterio = nn.CrossEntropyLoss()

    historia, mejor_f1, mejor_estado, sin_mejora = [], -1.0, None, 0
    t0 = time.perf_counter()

    for epoca in range(1, epocas + 1):
        modelo.train()
        perdida_train, aciertos, vistos, pred_tr, real_tr = 0.0, 0, 0, [], []
        for ids, lens, y in dl_train:
            ids, lens, y = ids.to(DEVICE), lens.to(DEVICE), y.to(DEVICE)
            logits = modelo(ids, lens)
            perdida = criterio(logits, y)
            opt.zero_grad(); perdida.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), 5.0)
            opt.step()

            perdida_train += perdida.item() * y.size(0); vistos += y.size(0)
            pred_tr.append(logits.argmax(1).cpu()); real_tr.append(y.cpu())

        logits_val, real_val = predecir(modelo, dl_val)
        f1_val = f1_score(real_val, logits_val.argmax(1), average="macro")
        f1_tr  = f1_score(torch.cat(real_tr), torch.cat(pred_tr), average="macro")
        historia.append({
            "epoca": epoca,
            "train_loss": perdida_train / vistos,
            "val_loss": criterio(logits_val, real_val).item(),
            "train_f1": f1_tr, "val_f1": f1_val,
        })

        if f1_val > mejor_f1:
            mejor_f1, sin_mejora = f1_val, 0
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia:
                print(f"   early stopping en la época {epoca} "
                      f"(sin mejora desde la {epoca - paciencia})")
                break

    modelo.load_state_dict(mejor_estado)
    seg = time.perf_counter() - t0
    n_par = sum(p.numel() for p in modelo.parameters())
    print(f"   [{etiqueta} · semilla {semilla}] mejor val macro-F1 {mejor_f1:.4f} "
          f"· {len(historia)} épocas · {seg:.0f}s · {n_par:,} parámetros")
    return modelo, pd.DataFrame(historia), seg, n_par

In [ ]:
SEMILLAS = (42, 7, 2024)

def correr_experimento(etiqueta, construir_modelo, semillas=SEMILLAS, **kw):
    """Entrena una arquitectura con varias semillas y devuelve todas las corridas."""
    print(f"── {etiqueta}")
    corridas = []
    for s in semillas:
        modelo, hist, seg, n_par = entrenar(construir_modelo, s, etiqueta=etiqueta, **kw)
        logits, reales = predecir(modelo, DataLoader(ds_val, batch_size=256, num_workers=0))
        pred = logits.argmax(1).numpy()
        m = evaluar(reales.numpy(), pred, nombre=f"{etiqueta}-s{s}", particion="val",
                    notas=f"{seg:.0f}s, {n_par:,} params", verbose=False)
        corridas.append({"semilla": s, "modelo": modelo, "historia": hist,
                         "pred": pred, "reales": reales.numpy(), "m": m,
                         "seg": seg, "n_par": n_par})
    f1s = [c["m"]["macro_f1"] for c in corridas]
    print(f"   → macro-F1 {np.mean(f1s):.4f} ± {np.std(f1s):.4f} "
          f"(min {min(f1s):.4f}, max {max(f1s):.4f})\n")
    return corridas

corridas_e1 = correr_experimento("E1-lstm-aleatoria",
                                 lambda: LSTMClasificador(len(vocab)))

### 5.4 Curvas de entrenamiento

Los notebooks guía corren una o dos épocas y nunca grafican la evolución, así que no pueden
mostrar si el modelo sobreajusta. Aquí se registran las cuatro series por época y se
superponen las tres semillas.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.7))

for i, c in enumerate(corridas_e1):
    h = c["historia"]
    primera = (i == 0)
    ax1.plot(h.epoca, h.train_loss, color=AZUL,    linewidth=1.7, alpha=.85,
             label="entrenamiento" if primera else None)
    ax1.plot(h.epoca, h.val_loss,   color=NARANJA, linewidth=1.7, alpha=.85,
             label="validación"    if primera else None)
    ax2.plot(h.epoca, h.train_f1,   color=AZUL,    linewidth=1.7, alpha=.85,
             label="entrenamiento" if primera else None)
    ax2.plot(h.epoca, h.val_f1,     color=NARANJA, linewidth=1.7, alpha=.85,
             label="validación"    if primera else None)
    mejor = h.val_f1.idxmax()
    ax2.plot(h.epoca[mejor], h.val_f1[mejor], "o", color=NARANJA, markersize=7,
             markeredgecolor="white", markeredgewidth=1.4, zorder=5)

ax1.set_xlabel("época"); ax1.set_ylabel("pérdida (entropía cruzada)")
ax1.set_title("Pérdida"); ax1.legend(loc="upper right")
ax1.grid(); ax1.set_axisbelow(True); limpiar(ax1)

techo_b1 = max(v["m"]["macro_f1"] for v in resultados_b1.values())
ax2.axhline(techo_b1, color=TINTA_2, linestyle="--", linewidth=1.2)
ax2.text(.03, techo_b1 - .028, "mejor B1 (TF-IDF balanceado)",
         transform=ax2.get_yaxis_transform(), ha="left", fontsize=8.5, color=TINTA_2,
         bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
ax2.set_xlabel("época"); ax2.set_ylabel("macro-F1")
ax2.set_title("Macro-F1  ·  el punto marca la época restaurada")
ax2.legend(loc="lower right"); ax2.grid(); ax2.set_axisbelow(True); limpiar(ax2)

fig.suptitle("E1 — tres semillas superpuestas", x=.005, ha="left",
             fontsize=11.5, fontweight="bold", color=TINTA)
plt.tight_layout(rect=[0, 0, 1, .92]); guardar(fig, "07-curvas-e1"); plt.show()

**Lectura.** Al revisar las curvas se nota que, aproximadamente desde la quinta época, el
modelo sigue mejorando en entrenamiento pero empeora en validación. La pérdida de
entrenamiento baja hasta 0,93, mientras que la de validación pasa de 1,51 a más de 1,60.
Esto indica sobreajuste: para el tamaño del conjunto de datos, una red con 2,69 millones de
parámetros tiene capacidad suficiente para aprender detalles propios del entrenamiento que
no se mantienen en datos nuevos.

Por esta razón usamos *early stopping* con la métrica de validación. Si se vigilara solamente
`train-loss`, como ocurre en el notebook guía, el entrenamiento continuaría aunque el
rendimiento sobre validación ya estuviera empeorando.

En el panel de macro-F1 también se observa que el resultado alto de entrenamiento no se
traslada a validación. La red llega a superar allí la referencia de TF-IDF, pero no lo hace
con datos que no ha visto. Además, las tres semillas producen resultados bastante distintos:
el macro-F1 varía entre 0,251 y 0,351. Por eso reportamos el promedio y la dispersión, en vez
de escoger únicamente la mejor corrida.

---

## 6. Contraste de resultados

### 6.1 Los tres modelos, lado a lado

In [ ]:
f1_e1 = [c["m"]["macro_f1"] for c in corridas_e1]
comparacion = pd.DataFrame([
    {"id": "B0", "modelo": "clase mayoritaria",
     "macro_f1": 0.0998, "sd": 0.0, "accuracy": 0.3325, "seg": 0.0, "params": 0},
    *[{"id": "B1", "modelo": k.replace("B1-", ""),
       "macro_f1": v["m"]["macro_f1"], "sd": 0.0,
       "accuracy": v["m"]["accuracy"], "seg": v["seg"], "params": np.nan}
      for k, v in resultados_b1.items()],
    {"id": "E1", "modelo": "LSTM embedding aleatoria",
     "macro_f1": float(np.mean(f1_e1)), "sd": float(np.std(f1_e1)),
     "accuracy": float(np.mean([c["m"]["accuracy"] for c in corridas_e1])),
     "seg": float(np.mean([c["seg"] for c in corridas_e1])),
     "params": corridas_e1[0]["n_par"]},
])
print(comparacion.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

mejor_b1 = max(resultados_b1.values(), key=lambda v: v["m"]["macro_f1"])
mejor_e1 = max(corridas_e1, key=lambda c: c["m"]["macro_f1"])
brecha = np.mean(f1_e1) - mejor_b1["m"]["macro_f1"]
print(f"\nH1 — LSTM menos el mejor TF-IDF: {brecha:+.4f} macro-F1")
print(f"     dispersión de E1 entre semillas: ±{np.std(f1_e1):.4f}")
print(f"     veredicto: H1 {'SE SOSTIENE' if brecha > 2*np.std(f1_e1) else 'NO SE SOSTIENE'}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.4, 3.8),
                               gridspec_kw={"width_ratios": [1.15, 1]})

# — izquierda: macro-F1 agregado de cada modelo
orden = comparacion.sort_values("macro_f1")
ax1.barh(range(len(orden)), orden.macro_f1, height=.62,
         color=[NARANJA if i == "E1" else AZUL for i in orden.id],
         xerr=orden.sd, error_kw={"ecolor": TINTA_2, "elinewidth": 1, "capsize": 3})
ax1.set_yticks(range(len(orden)))
ax1.set_yticklabels([f"{r.id} · {r.modelo}" for r in orden.itertuples()], fontsize=9)
for i, (v, s) in enumerate(zip(orden.macro_f1, orden.sd)):
    ax1.text(v + s + .015, i, f"{v:.3f}", va="center", fontsize=9, color=TINTA_2)
ax1.set_xlim(0, max(orden.macro_f1) * 1.25)
ax1.set_xlabel("macro-F1 en validación")
ax1.set_title("Ningún modelo neuronal hacía falta")
ax1.grid(axis="x"); ax1.set_axisbelow(True); limpiar(ax1)

# — derecha: dónde exactamente pierde la red
f1_por_clase_b1 = f1_score(df_val.y.values - 1, mejor_b1["pred"] - 1, average=None)
f1_por_clase_e1 = f1_score(mejor_e1["reales"], mejor_e1["pred"], average=None)
y = np.arange(5); ancho = .36
ax2.barh(y + ancho/2, f1_por_clase_b1, ancho, color=AZUL,    label="B1 · TF-IDF balanceado")
ax2.barh(y - ancho/2, f1_por_clase_e1, ancho, color=NARANJA, label="E1 · LSTM (mejor semilla)")
ax2.set_yticks(y); ax2.set_yticklabels(corto, fontsize=9)
ax2.set_xlabel("F1 por clase")
ax2.set_title("La red abandona las clases pequeñas")
ax2.legend(loc="lower right", fontsize=8.5)
ax2.grid(axis="x"); ax2.set_axisbelow(True); limpiar(ax2)

plt.tight_layout(); guardar(fig, "08-comparacion-modelos"); plt.show()

for c, a, b in zip(corto, f1_por_clase_e1, f1_por_clase_b1):
    print(f"{c:22s}  E1 {a:.3f}   B1 {b:.3f}   Δ {a-b:+.3f}")

### 6.2 Matrices de confusión

El panel derecho de la gráfica anterior ya adelanta dónde se pierde la comparación. Las
matrices lo muestran con todo detalle.

In [ ]:
# Ambos paneles en el mismo espacio de índices 0..4 (el corpus etiqueta 1..5).
paneles = [("B1 · TF-IDF + regresión logística", df_val.y.values - 1, mejor_b1["pred"] - 1),
           ("E1 · LSTM con embedding aleatoria", mejor_e1["reales"],  mejor_e1["pred"])]

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.6))
for ax, (titulo, y_true, y_pred) in zip(axes, paneles):
    C = confusion_matrix(y_true, y_pred, normalize="true")
    im = ax.imshow(C * 100, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(5)); ax.set_xticklabels(corto, rotation=32, ha="right", fontsize=8.5)
    ax.set_yticks(range(5)); ax.set_yticklabels(corto, fontsize=8.5)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{C[i, j]*100:.0f}", ha="center", va="center", fontsize=9,
                    color="white" if C[i, j] > .55 else TINTA)
    ax.set_title(titulo, fontsize=10.5)
    ax.set_xlabel("predicción"); ax.set_ylabel("clase real")
    for e in ("top", "right", "bottom", "left"):
        ax.spines[e].set_visible(False)

fig.suptitle("Matrices de confusión, normalizadas por fila (% de cada clase real)",
             x=.005, ha="left", fontsize=11.5, fontweight="bold", color=TINTA)
plt.tight_layout(rect=[0, 0, 1, .92]); guardar(fig, "09-matrices-confusion"); plt.show()

**Lectura.** La matriz de B1 conserva una diagonal clara: según la clase, acierta entre el
41% y el 68% de los ejemplos. En E1 el comportamiento cambia bastante. El modelo envía el
72% de los abstracts de *digestive* y el 73% de *nervous system* a la clase residual; por
eso sólo reconoce correctamente el 8% y el 1% de esas dos clases.

Nuestra interpretación es que la LSTM está favoreciendo la clase con más ejemplos. Esto es
coherente con la función de pérdida utilizada, porque la entropía cruzada sin ponderar no
penaliza de manera especial que una clase minoritaria quede casi sin predicciones. El
macro-F1 sí deja visible este problema al calcular el desempeño de cada clase por separado.

La comparación también requiere una precaución: el mejor B1 usa
`class_weight="balanced"`, mientras que E1 se entrenó sin ponderación. Por tanto, la brecha
de 0,245 mezcla el efecto de la arquitectura con el tratamiento del desbalance. Si se
comparan las dos versiones sin ponderar, TF-IDF obtiene 0,486 y E1 obtiene 0,285; la
diferencia sigue siendo amplia (0,201) y H1 continúa sin sostenerse.

A partir de este resultado, en los siguientes experimentos neuronales conviene utilizar una
pérdida ponderada por clase. Así podremos evaluar los cambios de embeddings y arquitectura
sin que el desbalance explique por sí solo una parte importante del resultado.

### 6.3 Contraste de las predicciones P1–P3

En 3.6 planteamos tres predicciones sobre la matriz de confusión a partir de la similitud
coseno entre centroides TF-IDF, antes de entrenar los modelos. Ahora comparamos esas
predicciones con los errores que realmente produjo E1.

In [ ]:
from scipy.stats import spearmanr, pearsonr

C_e1 = confusion_matrix(mejor_e1["reales"], mejor_e1["pred"], normalize="true")

# Tasa de confusión simétrica de cada par, frente a su similitud de centroides.
filas = []
for i in range(5):
    for j in range(i + 1, 5):
        filas.append({"par": f"{corto[i]} ↔ {corto[j]}",
                      "similitud": S[i, j],
                      "confusion": (C_e1[i, j] + C_e1[j, i]) / 2})
pares_df = pd.DataFrame(filas).sort_values("confusion", ascending=False)

rho, p_rho = spearmanr(pares_df.similitud, pares_df.confusion)
r, p_r = pearsonr(pares_df.similitud, pares_df.confusion)
print(pares_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nSpearman ρ = {rho:+.3f}  (p = {p_rho:.4f})")
print(f"Pearson  r = {r:+.3f}  (p = {p_r:.4f})")

# — las tres predicciones, una por una
err_por_clase = {corto[i]: (C_e1[i].sum() - C_e1[i, i]) + (C_e1[:, i].sum() - C_e1[i, i])
                 for i in range(5)}
peor_clase = max(err_por_clase, key=err_por_clase.get)
par_top = pares_df.iloc[0].par
par_min = pares_df.iloc[-1].par

print(f"\nP1 · clase que concentra más error : {peor_clase}")
print(f"     → {'CUMPLIDA' if peor_clase == 'general pathological' else 'FALLIDA'}")
print(f"P2 · par más confundido            : {par_top}")
print(f"     → {'CUMPLIDA' if set(par_top.split(' ↔ ')) == {'general pathological', 'nervous system'} else 'FALLIDA'}")
print(f"P3 · par mejor separado            : {par_min}")
print(f"     → {'CUMPLIDA' if set(par_min.split(' ↔ ')) == {'neoplasms', 'cardiovascular'} else 'FALLIDA'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.6))

involucra_general = pares_df.par.str.contains("general")
ax.scatter(pares_df.similitud[~involucra_general] * 100,
           pares_df.confusion[~involucra_general] * 100,
           s=130, color=AZUL, edgecolor="white", linewidth=1.6, zorder=3,
           label="pares entre clases anatómicas")
ax.scatter(pares_df.similitud[involucra_general] * 100,
           pares_df.confusion[involucra_general] * 100,
           s=130, color=NARANJA, edgecolor="white", linewidth=1.6, zorder=3,
           label="pares que incluyen la clase residual")

# recta de ajuste, sólo como guía visual de la tendencia, acotada al rango observado
b, a = np.polyfit(pares_df.similitud * 100, pares_df.confusion * 100, 1)
xs = np.linspace(pares_df.similitud.min() * 100, pares_df.similitud.max() * 100, 50)
ax.plot(xs, a + b * xs, color=TINTA_2, linestyle="--", linewidth=1.3, zorder=2)

# Se rotulan sólo los pares que la narración discute: los cuatro de la clase residual
# (a la izquierda de su punto, que están pegados al borde derecho) y el mejor separado.
# El resto forma un grupo compacto que la leyenda ya identifica.
for r_ in pares_df.itertuples():
    if "general" in r_.par:
        ax.annotate(r_.par, (r_.similitud * 100, r_.confusion * 100), ha="right",
                    textcoords="offset points", xytext=(-13, -4), fontsize=8, color=TINTA_2)
    elif r_.par.startswith("neoplasms ↔ cardio"):
        ax.annotate(r_.par + "  (P3)", (r_.similitud * 100, r_.confusion * 100), ha="left",
                    textcoords="offset points", xytext=(11, -17), fontsize=8, color=TINTA_2)

ax.set_xlabel("similitud coseno entre centroides TF-IDF  (§3.6, medida antes de entrenar)")
ax.set_ylabel("confusión simétrica de E1")
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_title(f"Lo medido antes de entrenar predice el error:  ρ de Spearman = {rho:+.2f}")
ax.set_xlim(54, 93); ax.set_ylim(-2.5, 45)
ax.legend(loc="upper left"); ax.grid(); ax.set_axisbelow(True); limpiar(ax)
plt.tight_layout(); guardar(fig, "10-prediccion-vs-confusion"); plt.show()

### 6.4 Qué dicen los números

En esta primera comparación, **H1 no se sostiene**. La LSTM obtiene en promedio 0,285 de
macro-F1, mientras que el mejor TF-IDF llega a 0,530. Como ese TF-IDF usa ponderación de
clases y E1 no, también revisamos la comparación sin ponderación: 0,486 frente a 0,285. La
diferencia sigue siendo de 0,201, mayor que la variación observada entre semillas (±0,047).
Además, E1 requiere 2,69 millones de parámetros y cerca de 26 segundos de GPU por corrida;
TF-IDF tarda unos 6 segundos en CPU.

El *accuracy* medio de E1 es 0,394, sólo seis puntos por encima del 0,333 obtenido al
predecir siempre la clase mayoritaria. Este valor aislado podría parecer aceptable, pero el
macro-F1 y la matriz de confusión muestran que el modelo casi no aprende algunas clases.

Una explicación posible es que, para este corpus, la presencia de términos médicos aporta
más a la clasificación que el orden de las palabras. Esto favorecería a TF-IDF, que utiliza
n-gramas directamente. Sin embargo, no afirmamos que la información secuencial sea
irrelevante: E1 todavía tiene limitaciones por la pérdida sin ponderar, los embeddings
aleatorios y el manejo del *padding*. Por eso interpretamos el resultado como evidencia a
favor de TF-IDF en esta configuración, no como una conclusión definitiva sobre todas las
LSTM.

**Dos de las tres predicciones se cumplen; P2 falla por un punto.** La correlación entre la
similitud de centroides —calculada sin entrenar nada— y la confusión real es de ρ = +0,891
(p = 0,0005), y la gráfica separa limpiamente los pares de la clase residual del resto.

- **P1 ✓** — *general pathological* concentra el error, como se predijo.
- **P2 ✗** — se predijo que el par más confundido sería *general* ↔ *nervous*; el real es
  *general* ↔ *digestive*, con 39,0% frente a 37,9%. Los dos primeros puestos están separados
  por **1,1 puntos**, muy por debajo de lo que esta metodología puede resolver. Lo honesto es
  registrarla como fallida: la predicción se hizo sobre el orden exacto y el orden exacto no
  se cumplió, aunque el grupo de cabeza sí se identificara bien.
- **P3 ✓** — *neoplasms* ↔ *cardiovascular* es el par mejor separado, con 1,4% de confusión.

Lo que sí queda demostrado es **el mecanismo**: el modelo se apoya en evidencia léxica, no en
estructura secuencial. Y eso cierra el círculo con H1 — si lo que decide es el vocabulario,
un modelo que sólo ve vocabulario no está en desventaja.

**El desbalance pesa más de lo que sugería su magnitud.** Ponderar las clases lleva el
macro-F1 de 0,486 a 0,530: **+4,4 puntos por un solo argumento**, la mejora más barata de
todo el notebook. Un desbalance de 3,2:1 parecía moderado y resultó no serlo, lo que refuerza
la decisión de §3.1 y añade un matiz: no basta con *medir* bien el desbalance, hay que
*corregirlo* en el modelo. La pérdida ponderada pasa a ser configuración por defecto en las
fases siguientes, no una variable a explorar.

**Lo que esto significa para las fases 4 y 5.** El experimento de embeddings se vuelve más
interesante, no menos: la pregunta ya no es sólo si los vectores de spaCy ayudan a la LSTM,
sino **si consiguen que la LSTM alcance siquiera a TF-IDF**, que le lleva 0,245 de ventaja.
La medición de §3.4 —un tercio de los tipos, y precisamente los discriminativos, sin
vector— predice que no bastará y que hará falta el brazo E4 con vectores biomédicos.

Y aparece una sospecha que antes no se podía formular: **una parte de esta derrota puede no
ser culpa de la arquitectura sino del diseño heredado**. E1 clasifica desde `hidden[-1]`, el
estado al que llega la LSTM tras recorrer las posiciones de relleno, que son el 31% del
total (§5.2). La ablación A2 de la fase 5 mide exactamente eso, y ahora tiene mucho más en
juego que cuando se planificó.

---

## 7. Conclusiones de la entrega

A partir del EDA y de los primeros modelos, nos quedamos con cinco conclusiones principales:

1. La métrica elegida cambia la lectura del problema. El modelo de clase mayoritaria logra
   0,33 de *accuracy*, pero su macro-F1 es apenas 0,10 porque ignora cuatro de las cinco
   categorías. Para este conjunto de datos, macro-F1 describe mejor el desempeño real.

2. La cobertura de spaCy es alta si se cuenta por token (97,2%), pero baja al contar tipos
   distintos (67,8%). Además, varias palabras sin vector son términos médicos útiles para
   separar las clases. Esto justifica probar embeddings especializados y no asumir que un
   modelo general cubre bien el dominio biomédico.

3. En esta primera comparación, TF-IDF funciona mejor que la LSTM. El mejor TF-IDF obtiene
   0,530 de macro-F1, mientras que E1 promedia 0,285. Incluso comparando las versiones sin
   ponderación, la ventaja sigue siendo de 0,201. Por ahora, H1 no se sostiene.

4. La matriz de confusión muestra que E1 favorece la categoría mayoritaria y reconoce muy
   pocos ejemplos de *digestive* y *nervous system*. Esto sugiere que los próximos modelos
   neuronales deben usar pérdida ponderada y una forma de manejar correctamente el
   *padding*.

5. Las semillas sí afectan el resultado: E1 varía entre 0,251 y 0,351 de macro-F1. Por eso
   una sola ejecución no es suficiente para comparar arquitecturas; mantendremos varias
   corridas y reportaremos su dispersión.

### Conclusión general

Para este caso, una arquitectura más compleja no produjo automáticamente un mejor modelo.
La información que separa las categorías parece ser principalmente léxica, y TF-IDF la
aprovecha mejor con menos costo. Sin embargo, todavía no podemos concluir que una LSTM no
sea útil: la versión evaluada usa embeddings aleatorios, pérdida sin ponderar y el último
estado oculto después del *padding*. Los siguientes experimentos deben corregir esas tres
limitaciones antes de hacer una comparación definitiva.

### Qué sigue

| Fase | Contenido |
|---|---|
| 4 | Embedding inicializada con spaCy: congelada (E2), ajustable (E3) y con vectores biomédicos (E4) — contrasta H2 y H3 |
| 5 | Ablaciones: BiLSTM, *pooling* enmascarado frente a `hidden[-1]`, `pack_padded_sequence`, barrido de `max_length` |
| 6 | Techo con encoder biomédico y demo interactiva |

En la fase 5 mediremos cuánto influye el uso de `hidden[-1]`, ya que el 31% de las
posiciones procesadas por E1 corresponde a relleno. Si al corregirlo mejora el macro-F1,
podremos separar mejor las limitaciones de la arquitectura de los problemas de esta
implementación concreta.

Todas las corridas quedan registradas en `resultados/experimentos.csv`:

In [ ]:
registro = pd.read_csv(REGISTRO).drop_duplicates(subset=["experimento", "particion"], keep="last")
print("Registro acumulado de experimentos:\n")
print(registro[["experimento", "particion", "macro_f1", "accuracy", "notas"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

---

### Referencias

- Monroe, B., Colaresi, M. & Quinn, K. (2008). *Fightin' Words: Lexical Feature Selection
  and Evaluation for Identifying the Content of Political Conflict.* Political Analysis 16(4).
- Schopf, T., Braun, D. & Matthes, F. (2022). *Evaluating Unsupervised Text Classification:
  Zero-shot and Similarity-based Approaches.* — origen del corpus `medical_abstracts`.
- Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory.* Neural Computation 9(8).